**Step 1 : Dataset Selection**  

**Labels : B-NP, I-NP, B-VP, etc.**  
**Dataset Used : CoNLL-2003**

**Step 2 : Install & Import Libraries**

In [ ]:
!pip install transformers datasets seqeval

from datasets import load_dataset
from transformers import AutoTokenizer

**Step 3 : Load Dataset**

In [ ]:
from datasets import load_dataset

# Load CoNLL-2003 (parquet-compatible version)
dataset = load_dataset("lhoestq/conll2003")

# Check structure
print(dataset)
print(dataset["train"][0])

**Step 4 : Tokenization**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

**Step 5 : Model Setup**

In [ ]:
from transformers import AutoModelForTokenClassification

# Define label list (NER tags)
label_list = [
    "O",
    "B-PER", "I-PER",
    "B-ORG", "I-ORG",
    "B-LOC", "I-LOC",
    "B-MISC", "I-MISC"
]

# Create mappings
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}

# Load model
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

print("Model loaded successfully")

**Step 6 : Training**

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

# Data collator (handles padding properly)
data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator
)

trainer.train()

**Step 7 : Evaluation**

In [ ]:
import numpy as np
from seqeval.metrics import classification_report

predictions, labels, _ = trainer.predict(tokenized_dataset["validation"])
predictions = np.argmax(predictions, axis=2)

true_predictions = [
    [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
    for pred, lab in zip(predictions, labels)
]

true_labels = [
    [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
    for pred, lab in zip(predictions, labels)
]

print(classification_report(true_labels, true_predictions))

**Step 8 : Example**

In [ ]:
from transformers import pipeline

nlp = pipeline("token-classification", model=model, tokenizer=tokenizer)

sentence = "John works at Google in California"
result = nlp(sentence)

print(result)

### Difference between POS Tagging and Chunking

POS Tagging assigns grammatical labels like noun, verb to each word, while chunking groups words into meaningful phrases like noun phrases and verb phrases.

### Challenges Faced

- Handling subword tokenization
- Aligning labels with tokens
- Managing special tokens (-100)

### Observations

- BERT captures context effectively
- Chunking is more complex than POS tagging

**Step 9 : Comparison**

| Feature    | POS Tagging | Chunking    |
| ---------- | ----------- | ----------- |
| Level      | Word        | Phrase      |
| Difficulty | Easy        | Medium      |
| Example    | Noun, Verb  | Noun Phrase |
